# FIT5201 Assignment 2 - Section 3: Unsupervised Learning
**Student Name**: [Yin Zhang]
**Student ID**: [[Student ID removed]]

## Question 1: Self-Taught Learning with Autoencoder

### I. Data preparation and protocol (5 marks)

In [ ]:
# I. Load data and prepare train/validation/test sets

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

%matplotlib inline

def save_show(path=None):
    """
    Save the current figure if a path is given, then show it in the notebook.
    This ensures figures appear in exported HTML/PDF files.
    """
    if path:
        plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()


# Load Task2C_labeled.csv, Task2C_unlabeled.csv, and Task2C_test.csv
labeled_df = pd.read_csv('Dataset_S2_2025/Task2C_labeled.csv')
unlabeled_df = pd.read_csv('Dataset_S2_2025/Task2C_unlabeled.csv')
test_df = pd.read_csv('Dataset_S2_2025/Task2C_test.csv')

print(f"Labeled data shape: {labeled_df.shape}")
print(f"Unlabeled data shape: {unlabeled_df.shape}")
print(f"Test data shape: {test_df.shape}")

X_labeled = labeled_df.drop(columns=['label']).values
y_labeled = labeled_df['label'].values

X_unlabeled = unlabeled_df.values

X_test = test_df.drop(columns=['label']).values
y_test = test_df['label'].values

print(f"\nX_labeled shape: {X_labeled.shape}")
print(f"y_labeled shape: {y_labeled.shape}")
print(f"X_unlabeled shape: {X_unlabeled.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# Split labelled data into training and validation sets (80/20 split)
from IPython.display import Markdown, display

X_train_labeled, X_val_labeled, y_train_labeled, y_val_labeled = train_test_split(
    X_labeled,
    y_labeled,
    test_size=0.2,
    random_state=42,
    stratify=y_labeled
)

print(f"Training labelled data shape: {X_train_labeled.shape}")
print(f"Validation labelled data shape: {X_val_labeled.shape}")
print(f"Training labels shape: {y_train_labeled.shape}")
print(f"Validation labels shape: {y_val_labeled.shape}")

# Scale pixel intensities to [0, 1] for stable neural-network training
PIXEL_SCALE = 255.0

X_train_labeled = X_train_labeled.astype(float) / PIXEL_SCALE
X_val_labeled = X_val_labeled.astype(float) / PIXEL_SCALE
X_unlabeled = X_unlabeled.astype(float) / PIXEL_SCALE
X_test = X_test.astype(float) / PIXEL_SCALE

print(f"Scaled features — train: {X_train_labeled.shape}, val: {X_val_labeled.shape}, test: {X_test.shape}")

protocol_md = f"""
### I. Data split and usage protocol

The labelled dataset was split into **80% training** and **20% validation** using stratified sampling.

- Labelled training set: **{X_train_labeled.shape[0]} samples**
- Validation set: **{X_val_labeled.shape[0]} samples**
- Unlabelled set: **{X_unlabeled.shape[0]} samples**
- Test set: **{X_test.shape[0]} samples**

The data is used as follows:

1. **Autoencoder training:** use `X_train_labeled` together with `X_unlabeled`.
2. **Classifier training:** use only `X_train_labeled` and `y_train_labeled`.
3. **Model selection:** use `X_val_labeled` and `y_val_labeled`.
4. **Final evaluation:** use `X_test` and `y_test`.

This protocol avoids using validation or test labels during training. The unlabelled data is used only to help the autoencoder learn a more general representation of the input distribution.
"""

display(Markdown(protocol_md))

### II. Autoencoder representation study (8 marks)

In [ ]:
# Single-hidden-layer autoencoder implementation
class Autoencoder:
    def __init__(self, input_size, hidden_size, learning_rate=0.01, epochs=100):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.learning_rate = learning_rate
        self.epochs = epochs

        # Encoder parameters
        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros(hidden_size)

        # Decoder parameters
        self.W2 = np.random.randn(hidden_size, input_size) * 0.01
        self.b2 = np.zeros(input_size)

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def sigmoid_derivative(self, z):
        s = self.sigmoid(z)
        return s * (1 - s)

    def forward(self, X):
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = self.sigmoid(self.Z1)

        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        self.A2 = self.sigmoid(self.Z2)

        return self.A2

    def backward(self, X):
        m = X.shape[0]

        # Reconstruction loss gradient
        dZ2 = (self.A2 - X) * self.sigmoid_derivative(self.Z2)
        dW2 = (1 / m) * np.dot(self.A1.T, dZ2)
        db2 = (1 / m) * np.sum(dZ2, axis=0)

        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * self.sigmoid_derivative(self.Z1)
        dW1 = (1 / m) * np.dot(X.T, dZ1)
        db1 = (1 / m) * np.sum(dZ1, axis=0)

        return dW1, db1, dW2, db2

    def fit(self, X):
        for epoch in range(self.epochs):
            self.forward(X)

            dW1, db1, dW2, db2 = self.backward(X)

            self.W1 -= self.learning_rate * dW1
            self.b1 -= self.learning_rate * db1
            self.W2 -= self.learning_rate * dW2
            self.b2 -= self.learning_rate * db2

        return self

    def encode(self, X):
        self.forward(X)
        return self.A1

    def reconstruct(self, X):
        return self.forward(X)

    def reconstruction_error(self, X):
        """
        Average reconstruction error required by the assignment:
        (1/N) * sum ||x_i - x_hat_i||_2
        """
        reconstructed = self.reconstruct(X)
        return np.mean(np.linalg.norm(X - reconstructed, axis=1))

    def get_hidden_representation(self, X):
        return self.encode(X)

In [ ]:
# Train autoencoders with different hidden sizes

hidden_sizes = [20, 60, 100, 140, 180, 220]
reconstruction_errors = []
autoencoders = {}

# Combine labelled training data and unlabelled data for autoencoder training
X_autoencoder = np.vstack([X_train_labeled, X_unlabeled])

print("Training autoencoders with different hidden sizes...")
print(f"Autoencoder training data shape: {X_autoencoder.shape}")

for hidden_size in hidden_sizes:
    autoencoder = Autoencoder(
        input_size=X_autoencoder.shape[1],
        hidden_size=hidden_size,
        learning_rate=0.01,
        epochs=100
    )

    autoencoder.fit(X_autoencoder)

    # Assignment formula: average L2 reconstruction error
    reconstruction_error = autoencoder.reconstruction_error(X_val_labeled)

    reconstruction_errors.append(reconstruction_error)
    autoencoders[hidden_size] = autoencoder

    print(f"Hidden size {hidden_size}: validation average reconstruction error = {reconstruction_error:.6f}")

print("\nValidation reconstruction errors:")
for hidden_size, error in zip(hidden_sizes, reconstruction_errors):
    print(f"H={hidden_size}: average reconstruction error = {error:.6f}")



In [ ]:
from IPython.display import Markdown, display

# Plot reconstruction error vs hidden size
plt.figure(figsize=(10, 6))
plt.plot(hidden_sizes, reconstruction_errors, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Hidden Layer Size (H)', fontsize=12)
plt.ylabel('Validation Average Reconstruction Error', fontsize=12)
plt.title('Autoencoder Reconstruction Error vs Hidden Size', fontsize=14)
plt.grid(True, alpha=0.3)
save_show('task2c_reconstruction_error.png')

print("Reconstruction error plot saved as 'task2c_reconstruction_error.png'")

errs = np.array(reconstruction_errors)
deltas = np.diff(errs)

best_recon_h = hidden_sizes[int(np.argmin(errs))]
worst_recon_h = hidden_sizes[int(np.argmax(errs))]

n_decrease = int(np.sum(deltas < 0))
n_increase = int(np.sum(deltas > 0))

recon_md = f"""### II. Autoencoder reconstruction trend

The validation reconstruction error was computed using the assignment formula:

$\\frac{{1}}{{N}} \\sum_i \\|x_i - \\hat{{x}}_i\\|_2$

- Lowest validation reconstruction error: **H = {best_recon_h}** with error **{errs.min():.6f}**
- Highest validation reconstruction error: **H = {worst_recon_h}** with error **{errs.max():.6f}**
- As H increases, the reconstruction error shows **{n_decrease} decreases** and **{n_increase} increases** between adjacent hidden sizes.

"""

if n_decrease == len(deltas) and n_increase == 0:
    recon_md += (
        "The reconstruction error decreases monotonically as H grows. "
        "This suggests that a larger hidden representation gives the autoencoder more capacity to reconstruct the input data."
    )
elif n_increase == len(deltas) and n_decrease == 0:
    recon_md += (
        "The reconstruction error increases as H grows in this run. "
        "This is unusual for an autoencoder and may indicate insufficient training, optimisation instability, or sensitivity to random initialisation."
    )
else:
    recon_md += (
        "The curve is non-monotonic. This means that increasing H does not always reduce validation reconstruction error. "
        "Possible reasons include optimisation noise, limited epochs, and sensitivity to random initialisation."
    )

display(Markdown(recon_md))

### III. Baseline and self-taught classifiers (12 marks)

In [ ]:
# 3-layer Neural Network classifier for 10 digit classes
NUM_CLASSES = 10

def to_one_hot(y, num_classes=NUM_CLASSES):
    y = np.asarray(y, dtype=int).reshape(-1)
    one_hot = np.zeros((len(y), num_classes))
    one_hot[np.arange(len(y)), y] = 1.0
    return one_hot


class ClassifierNN:
    def __init__(self, input_size, hidden_size, output_size=NUM_CLASSES, lambda_reg=0.001, learning_rate=0.01):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.lambda_reg = lambda_reg
        self.learning_rate = learning_rate

        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros(hidden_size)

        self.W2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros(output_size)

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def sigmoid_derivative(self, z):
        s = self.sigmoid(z)
        return s * (1 - s)

    def softmax(self, z):
        z_shifted = z - np.max(z, axis=1, keepdims=True)
        exp_z = np.exp(z_shifted)
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def forward(self, X):
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = self.sigmoid(self.Z1)

        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        self.A2 = self.softmax(self.Z2)

        return self.A2

    def backward(self, X, y, output):
        m = X.shape[0]
        y_oh = to_one_hot(y, self.output_size) if y.ndim == 1 else y

        # Softmax + cross-entropy gradient
        dZ2 = output - y_oh

        dW2 = (1 / m) * np.dot(self.A1.T, dZ2) + (self.lambda_reg / m) * self.W2
        db2 = (1 / m) * np.sum(dZ2, axis=0)

        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * self.sigmoid_derivative(self.Z1)

        dW1 = (1 / m) * np.dot(X.T, dZ1) + (self.lambda_reg / m) * self.W1
        db1 = (1 / m) * np.sum(dZ1, axis=0)

        return dW1, db1, dW2, db2

    def fit(self, X, y, epochs=1000, batch_size=32, early_stopping=False, patience=10, X_val=None, y_val=None):
        m = X.shape[0]

        best_W1, best_b1 = None, None
        best_W2, best_b2 = None, None
        best_val_error = float('inf')
        patience_counter = 0

        for epoch in range(epochs):
            indices = np.random.permutation(m)
            X_shuffled = X[indices]
            y_shuffled = y[indices]

            for i in range(0, m, batch_size):
                X_batch = X_shuffled[i:i + batch_size]
                y_batch = y_shuffled[i:i + batch_size]

                output = self.forward(X_batch)
                dW1, db1, dW2, db2 = self.backward(X_batch, y_batch, output)

                self.W1 -= self.learning_rate * dW1
                self.b1 -= self.learning_rate * db1
                self.W2 -= self.learning_rate * dW2
                self.b2 -= self.learning_rate * db2

            if early_stopping and X_val is not None and y_val is not None:
                val_error = evaluate_classifier(self, X_val, y_val)

                if val_error < best_val_error:
                    best_val_error = val_error
                    best_W1 = self.W1.copy()
                    best_b1 = self.b1.copy()
                    best_W2 = self.W2.copy()
                    best_b2 = self.b2.copy()
                    patience_counter = 0
                else:
                    patience_counter += 1

                if patience_counter >= patience:
                    break

        if early_stopping and best_W1 is not None:
            self.W1 = best_W1
            self.b1 = best_b1
            self.W2 = best_W2
            self.b2 = best_b2

        return self

    def predict(self, X):
        probs = self.forward(X)
        return np.argmax(probs, axis=1)


def evaluate_classifier(nn, X, y):
    return np.mean(nn.predict(X) != y)

In [ ]:
# Train baseline classifiers using original input features

print("Training baseline classifiers...")

baseline_errors = {}
baseline_classifiers = {}

for hidden_size in hidden_sizes:
    classifier = ClassifierNN(
        input_size=X_train_labeled.shape[1],
        hidden_size=hidden_size,
        output_size=NUM_CLASSES,
        lambda_reg=0.001,
        learning_rate=0.01
    )

    classifier.fit(
        X_train_labeled,
        y_train_labeled,
        epochs=1000,
        batch_size=32,
        early_stopping=True,
        patience=10,
        X_val=X_val_labeled,
        y_val=y_val_labeled
    )

    val_error = evaluate_classifier(classifier, X_val_labeled, y_val_labeled)

    baseline_errors[hidden_size] = val_error
    baseline_classifiers[hidden_size] = classifier

    print(f"Baseline H={hidden_size}: validation error = {val_error:.4f}")

best_hidden_baseline = min(baseline_errors, key=baseline_errors.get)

print(
    f"\nBest baseline classifier: H={best_hidden_baseline}, "
    f"validation error = {baseline_errors[best_hidden_baseline]:.4f}"
)

In [ ]:
# Train self-taught classifiers using original features + autoencoder hidden representations

print("\nTraining self-taught classifiers...")

self_taught_errors = {}
self_taught_classifiers = {}

for hidden_size in hidden_sizes:
    autoencoder = autoencoders[hidden_size]

    # Get hidden representations from the autoencoder
    X_train_hidden = autoencoder.get_hidden_representation(X_train_labeled)
    X_val_hidden = autoencoder.get_hidden_representation(X_val_labeled)

    # Concatenate original features with learned hidden representations
    X_train_combined = np.hstack([X_train_labeled, X_train_hidden])
    X_val_combined = np.hstack([X_val_labeled, X_val_hidden])

    classifier = ClassifierNN(
        input_size=X_train_combined.shape[1],
        hidden_size=hidden_size,
        output_size=NUM_CLASSES,
        lambda_reg=0.001,
        learning_rate=0.01
    )

    classifier.fit(
        X_train_combined,
        y_train_labeled,
        epochs=1000,
        batch_size=32,
        early_stopping=True,
        patience=10,
        X_val=X_val_combined,
        y_val=y_val_labeled
    )

    val_error = evaluate_classifier(classifier, X_val_combined, y_val_labeled)

    self_taught_errors[hidden_size] = val_error
    self_taught_classifiers[hidden_size] = classifier

    print(f"Self-taught H={hidden_size}: validation error = {val_error:.4f}")

best_hidden_self_taught = min(self_taught_errors, key=self_taught_errors.get)

print(
    f"\nBest self-taught classifier: H={best_hidden_self_taught}, "
    f"validation error = {self_taught_errors[best_hidden_self_taught]:.4f}"
)

In [ ]:
# Evaluate selected models on the test set

print("\nEvaluating selected models on the test set...")

# Baseline classifier test error
best_baseline_model = baseline_classifiers[best_hidden_baseline]
test_error_baseline = evaluate_classifier(
    best_baseline_model,
    X_test,
    y_test
)

# Self-taught classifier test error
best_self_taught_model = self_taught_classifiers[best_hidden_self_taught]
best_autoencoder = autoencoders[best_hidden_self_taught]

X_test_hidden = best_autoencoder.get_hidden_representation(X_test)
X_test_combined = np.hstack([X_test, X_test_hidden])

test_error_self_taught = evaluate_classifier(
    best_self_taught_model,
    X_test_combined,
    y_test
)

print(f"\nSelected baseline classifier:")
print(f"H = {best_hidden_baseline}")
print(f"Validation error = {baseline_errors[best_hidden_baseline]:.4f}")
print(f"Test error = {test_error_baseline:.4f}")

print(f"\nSelected self-taught classifier:")
print(f"H = {best_hidden_self_taught}")
print(f"Validation error = {self_taught_errors[best_hidden_self_taught]:.4f}")
print(f"Test error = {test_error_self_taught:.4f}")

print("\nValidation Errors:")
print("=" * 60)
print(f"{'H':<8} {'Baseline Val':<18} {'Self-Taught Val':<18}")
print("=" * 60)

for hidden_size in hidden_sizes:
    print(
        f"{hidden_size:<8} "
        f"{baseline_errors[hidden_size]:<18.4f} "
        f"{self_taught_errors[hidden_size]:<18.4f}"
    )

print("=" * 60)

### IV. Analysis (10 marks)

In [ ]:
from IPython.display import Markdown, display

def save_show(path=None):
    if path:
        plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()


# Plot validation errors for baseline and self-taught classifiers


plt.figure(figsize=(10, 6))

plt.plot(
    hidden_sizes,
    [baseline_errors[h] for h in hidden_sizes],
    'bo-',
    linewidth=2,
    markersize=8,
    label='Baseline'
)

plt.plot(
    hidden_sizes,
    [self_taught_errors[h] for h in hidden_sizes],
    'ro-',
    linewidth=2,
    markersize=8,
    label='Self-Taught'
)

plt.xlabel('Hidden Layer Size (H)', fontsize=12)
plt.ylabel('Validation Classification Error', fontsize=12)
plt.title('Baseline vs Self-Taught Classifier Performance', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)

save_show('task2c_baseline_vs_self_taught.png')

print("Validation error plot saved as 'task2c_baseline_vs_self_taught.png'")


# Helper function: repeat training with different random seeds


def evaluate_with_initializations(
    input_size,
    hidden_size,
    X_train,
    y_train,
    X_eval,
    y_eval,
    X_val,
    y_val,
    n_runs=3
):
    errors = []

    for seed in range(n_runs):
        np.random.seed(seed)

        nn_copy = ClassifierNN(
            input_size=input_size,
            hidden_size=hidden_size,
            output_size=NUM_CLASSES,
            lambda_reg=0.001,
            learning_rate=0.01
        )

        nn_copy.fit(
            X_train,
            y_train,
            epochs=1000,
            batch_size=32,
            early_stopping=True,
            patience=10,
            X_val=X_val,
            y_val=y_val
        )

        error = evaluate_classifier(nn_copy, X_eval, y_eval)
        errors.append(error)

    return np.mean(errors), np.std(errors), errors


# Prepare self-taught features using the selected autoencoder


ae_best = autoencoders[best_hidden_self_taught]

X_train_st = np.hstack([
    X_train_labeled,
    ae_best.get_hidden_representation(X_train_labeled)
])

X_val_st = np.hstack([
    X_val_labeled,
    ae_best.get_hidden_representation(X_val_labeled)
])

X_test_st = np.hstack([
    X_test,
    ae_best.get_hidden_representation(X_test)
])



# Repeat selected baseline classifier with 3 random initialisations


test_error_baseline_mean, test_error_baseline_std, baseline_seed_errors = evaluate_with_initializations(
    input_size=X_train_labeled.shape[1],
    hidden_size=best_hidden_baseline,
    X_train=X_train_labeled,
    y_train=y_train_labeled,
    X_eval=X_test,
    y_eval=y_test,
    X_val=X_val_labeled,
    y_val=y_val_labeled,
    n_runs=3
)



# Repeat selected self-taught classifier with 3 random initialisations


test_error_self_taught_mean, test_error_self_taught_std, self_taught_seed_errors = evaluate_with_initializations(
    input_size=X_train_st.shape[1],
    hidden_size=best_hidden_self_taught,
    X_train=X_train_st,
    y_train=y_train_labeled,
    X_eval=X_test_st,
    y_eval=y_test,
    X_val=X_val_st,
    y_val=y_val_labeled,
    n_runs=3
)


print("\n=== Final test summary ===")
print(f"Selected baseline H: {best_hidden_baseline}")
print(f"Selected self-taught H: {best_hidden_self_taught}")

print(f"\nSingle selected baseline test error: {test_error_baseline:.4f}")
print(f"Single selected self-taught test error: {test_error_self_taught:.4f}")

print(f"\nBaseline 3-seed test errors: {baseline_seed_errors}")
print(f"Baseline mean ± std: {test_error_baseline_mean:.4f} ± {test_error_baseline_std:.4f}")

print(f"\nSelf-taught 3-seed test errors: {self_taught_seed_errors}")
print(f"Self-taught mean ± std: {test_error_self_taught_mean:.4f} ± {test_error_self_taught_std:.4f}")



# Build final analysis markdown


if test_error_self_taught < test_error_baseline:
    test_line = (
        f"On the held-out test set, the selected self-taught classifier achieves lower error "
        f"(**{test_error_self_taught:.4f}**) than the selected baseline classifier "
        f"(**{test_error_baseline:.4f}**)."
    )
elif test_error_self_taught > test_error_baseline:
    test_line = (
        f"On the held-out test set, the selected baseline classifier achieves lower error "
        f"(**{test_error_baseline:.4f}**) than the selected self-taught classifier "
        f"(**{test_error_self_taught:.4f}**)."
    )
else:
    test_line = (
        f"The selected baseline and self-taught classifiers achieve the same test error "
        f"(**{test_error_baseline:.4f}**)."
    )


# More cautious stability interpretation
if test_error_self_taught_mean < test_error_baseline_mean:
    if test_error_self_taught_std > test_error_baseline_std:
        stability_note = (
            "Across 3 random initialisations, the self-taught classifier has a lower mean test error, "
            "but its standard deviation is larger than the baseline. Therefore, the self-taught result is less stable, "
            "and the single-run comparison should be interpreted cautiously."
        )
    else:
        stability_note = (
            "Across 3 random initialisations, the self-taught classifier has a lower mean test error and comparable or lower standard deviation. "
            "This suggests that the self-taught method gives a more reliable improvement in this experiment."
        )

elif test_error_self_taught_mean > test_error_baseline_mean:
    if test_error_self_taught_std > test_error_baseline_std:
        stability_note = (
            "Across 3 random initialisations, the baseline classifier has a lower mean test error and the self-taught classifier has higher variability. "
            "This suggests that the self-taught representation does not provide a stable improvement in this run."
        )
    else:
        stability_note = (
            "Across 3 random initialisations, the baseline classifier has a lower mean test error. "
            "Although the self-taught classifier is relatively stable, it does not outperform the baseline on average."
        )

else:
    stability_note = (
        "Across 3 random initialisations, the two methods have essentially the same mean test error. "
        "The standard deviations should be used to judge which method is more stable."
    )


best_recon_h = hidden_sizes[int(np.argmin(reconstruction_errors))]

if best_recon_h == best_hidden_self_taught:
    recon_line = (
        f"The lowest reconstruction error and the best self-taught validation error both occur at "
        f"**H = {best_recon_h}**. This suggests that the representation that reconstructs the input best "
        "also helps classification on the validation set."
    )
else:
    recon_line = (
        f"The lowest reconstruction error occurs at **H = {best_recon_h}**, while the best self-taught "
        f"validation error occurs at **H = {best_hidden_self_taught}**. This shows that better reconstruction "
        "does not always directly imply better classification performance."
    )


val_gain = baseline_errors[best_hidden_self_taught] - self_taught_errors[best_hidden_self_taught]

if val_gain > 0:
    unlabeled_line = (
        f"At the selected self-taught hidden size H = {best_hidden_self_taught}, the self-taught classifier "
        f"improves validation error over the baseline by **{val_gain:.4f}**. This suggests that the hidden "
        "representation learned from unlabelled data provides useful additional information."
    )
elif val_gain < 0:
    unlabeled_line = (
        f"At the selected self-taught hidden size H = {best_hidden_self_taught}, the self-taught classifier "
        f"has validation error **{-val_gain:.4f}** higher than the baseline. In this run, the representation "
        "learned from unlabelled data does not clearly improve classification."
    )
else:
    unlabeled_line = (
        f"At the selected self-taught hidden size H = {best_hidden_self_taught}, the baseline and self-taught "
        "validation errors are the same. This means that the unlabelled-data representation does not produce "
        "a clear validation improvement at this selected H."
    )


analysis_md = f"""### IV. Analysis

1. **Final test-set performance:**  
{test_line}

2. **Stability across random initialisations:**  
The baseline classifier obtains **{test_error_baseline_mean:.4f} ± {test_error_baseline_std:.4f}** over 3 random initialisations.  
The self-taught classifier obtains **{test_error_self_taught_mean:.4f} ± {test_error_self_taught_std:.4f}** over 3 random initialisations.  
{stability_note}

3. **Reconstruction versus classification:**  
{recon_line}

4. **Effect of unlabelled data:**  
{unlabeled_line}

5. **Validation results used for model selection:**

| H | Baseline validation error | Self-taught validation error | Average reconstruction error |
|---|---:|---:|---:|"""

for hs in hidden_sizes:
    recon_error = reconstruction_errors[hidden_sizes.index(hs)]
    analysis_md += (
        f"\n| {hs} | {baseline_errors[hs]:.4f} | "
        f"{self_taught_errors[hs]:.4f} | {recon_error:.6f} |"
    )

analysis_md += f"""

6. **Selected final models:**

| Model | Selected H | Validation error | Test error |
|---|---:|---:|---:|
| Baseline classifier | {best_hidden_baseline} | {baseline_errors[best_hidden_baseline]:.4f} | {test_error_baseline:.4f} |
| Self-taught classifier | {best_hidden_self_taught} | {self_taught_errors[best_hidden_self_taught]:.4f} | {test_error_self_taught:.4f} |

The per-H comparison uses validation errors only. The test set is used only for the final evaluation of the selected baseline and self-taught models.
"""

display(Markdown(analysis_md))